# Tests — LLM (génération) + TTS (synthèse vocale FR)

**Objectif :** compléter le "G" du RAG avec un LLM générateur, puis synthétiser la réponse en audio.

**Modèle LLM utilisé :** Llama 3.2 (Meta)

## 0. Setup

In [ ]:
# Setup, détection Colab/local

from pathlib import Path
import sys
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, display


try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path("/content/drive/MyDrive/noo-far-pipeline")
    !pip install chromadb rank-bm25 --quiet
else:
    PROJECT_ROOT = Path(r"C:\dev\noo-far-pipeline")
    
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Projet : {PROJECT_ROOT} | Sur Colab : {ON_COLAB}")

In [ ]:
from huggingface_hub import login
login(token="Mon_Token_HF")

## 1. Chargement du LLM

In [ ]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

llm = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.2-3B-Instruct",  # variante légère, cohérente avec un GPU T4
    device=device,
    torch_dtype=torch.float16 if device == 0 else torch.float32
)

## 2. Construction du prompt RAG

In [ ]:
from rag.retriever import retrieve

sys.path.insert(0, str(PROJECT_ROOT))

question = "Quand vacciner mes vaches ?"
passages = retrieve(question, mode="semantic")  # semantic, suite à la décision du J2

In [ ]:
def build_prompt(question, passages):
    contexte = "\n\n".join(f"[Extrait {i+1}] {doc}" for i, (doc, meta) in enumerate(passages))
    prompt = f"""Tu es un assistant vocal qui aide des éleveurs laitiers sénégalais.
Réponds à la question UNIQUEMENT à partir des extraits fournis ci-dessous.
Si l'information n'est pas dans les extraits, dis clairement que tu ne sais pas — n'invente rien.
Réponds en français, de façon concise et orale (pas de listes à puces, une réponse qu'on peut lire à voix haute).

Extraits :
{contexte}

Question : {question}

Réponse :"""
    return prompt

In [ ]:
prompt = build_prompt(question, passages)
result = llm(prompt, max_new_tokens=150, temperature=0.3, do_sample=True)
reponse = result[0]["generated_text"][len(prompt):].strip()  # retirer le prompt de la sortie
print(reponse)

## 4. Vérification d'ancrage (anti-hallucination)


### Test manuel — poser une question hors périmètre


In [ ]:
question_hors_sujet = "Quel est le prix du pétrole aujourd'hui ?"
passages_vides = retrieve(question_hors_sujet, mode="semantic")
prompt_test = build_prompt(question_hors_sujet, passages_vides)
result = llm(prompt_test, max_new_tokens=150, temperature=0.3)
print(result[0]["generated_text"][len(prompt_test):].strip())

### Vérification de cohérence simple

In [ ]:
def verifier_ancrage(reponse, passages, seuil=0.3):
    """Vérifie grossièrement si la réponse partage du vocabulaire avec les passages."""
    mots_reponse = set(reponse.lower().split())
    mots_passages = set(" ".join(doc for doc, meta in passages).lower().split())
    intersection = mots_reponse & mots_passages
    ratio = len(intersection) / max(len(mots_reponse), 1)
    return ratio >= seuil, ratio

In [ ]:
est_ancre, ratio = verifier_ancrage(reponse, passages)
print(f"Réponse ancrée : {est_ancre} (ratio de vocabulaire partagé : {ratio:.2f})")

## 5. Synthèse vocale (TTS)


In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch

tts_model = VitsModel.from_pretrained("facebook/mms-tts-fra")
tts_tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-fra")

if torch.cuda.is_available():
    tts_model = tts_model.to("cuda")

def synthesize_raw(texte):
    inputs = tts_tokenizer(texte, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        output = tts_model(**inputs).waveform
    return output.cpu().numpy().squeeze(), tts_model.config.sampling_rate

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

audio, sr = synthesize_raw(reponse)
sf.write("reponse_test.wav", audio, sr)
display(Audio(audio, rate=sr))

## 6. Test sur plusieurs questions


In [ ]:
questions_test = [
    "Quand vacciner mes vaches ?",
    "Quelle alimentation pour une vache en lactation ?",
    "Ma vache n'arrive pas à se reproduire",
]

for q in questions_test:
    passages = retrieve(q, mode="semantic")
    prompt = build_prompt(q, passages)
    result = llm(prompt, max_new_tokens=150, temperature=0.3)
    reponse = result[0]["generated_text"][len(prompt):].strip()

    print(f"\n{'='*60}\nQ: {q}\nR: {reponse}\n{'='*60}")
    audio, sr = synthesize_raw(reponse)
    display(Audio(audio, rate=sr))

## Synthèse — Observations

**LLM utilisé :** meta-llama/Llama-3.2-3B-Instruct (accès Meta initialement en attente `pending`, validé en
cours de journée).

**Génération LLM — 3 questions testées :**
- Q1 "Quand vacciner mes vaches ?" : fidèle à vaccination.md, léger décalage de pertinence (ouvre sur une
  info annexe "attendre le rétablissement" avant de répondre directement à la question posée).
- Q4 "Quelle alimentation pour une vache en lactation ?" : fidèle à alimentation.md (rations, ajustement
  selon poids/état physiologique).
- Q5 "Ma vache n'arrive pas à se reproduire" : fidèle à reproduction.md (âge de mise à la reproduction,
  24-36 mois).

**Test anti-hallucination (question hors périmètre) :** réponse correcte — *"Je ne sais pas. (Pas
d'information dans les extraits fournis.)"* Le garde-fou du prompt fonctionne comme prévu.

**Vérification d'ancrage :** ratio de 0.97 sur Q1 — très élevé, cohérent avec l'absence d'hallucination
détectée à la lecture.

**TTS MMS-FR :** voix peu naturelle, intonations parfois mal placées — comportement attendu du modèle
(généraliste multilingue, non optimisé par langue). Intelligibilité globalement correcte malgré la prosodie
imparfaite. Piste retenue : simplifier la ponctuation des réponses avant synthèse. À mesurer plus
rigoureusement au MOS baseline (S2 pour le wolof, potentiellement plus dégradé encore).

**Lien avec J2 (retrieval) :** les 3 questions testées ici (Q1, Q4, Q5) correspondent exactement aux 3
questions où le retrieval sémantique avait réussi en J2 — cohérence confirmée entre bonne récupération et
bonne génération. Les questions en échec en J2 (Q2 fièvre, Q3 traite) n'ont pas été testées en génération,
à faire lors d'une prochaine session pour voir si le LLM peut compenser un retrieval imparfait ou si l'erreur
se propage.

**Non traité aujourd'hui, reporté :**
- Nettoyage de ponctuation avant TTS
- Amélioration du contenu des fiches sante_animale.md et production_laitiere.md (identifié en J2)
- Pondération de la fusion hybride BM25/sémantique
- Test génération sur Q2/Q3 (retrieval en échec en J2)